In [6]:
from preprocessing.data_preprocessing import get_processed_data
from preprocessing.utils import clean_chunk

PDF_PATH = '../data/coffee_processing.pdf'

In [9]:
final_data = get_processed_data(PDF_PATH)
print(final_data)

Total blocks: 21 | {'text': 17, 'table': 2, 'image': 2}
  [table] p2 → summarized & stored
  [image] p2 → summarized & stored
  [image] p2 → summarized & stored
  [table] p4 → summarized & stored
{'texts': ['A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplis

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)
text_chunks = text_splitter.create_documents(final_data['texts'])


cleaned_documents = []

for i, chunk in enumerate(text_chunks):
    raw_text = chunk.page_content
    cleaned_text = clean_chunk(raw_text)
    
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)

for i, text_summary in enumerate(final_data['images']):
    cleaned_text = clean_chunk(text_summary)
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)

for i, text_summary in enumerate(final_data['tables']):
    cleaned_text = clean_chunk(text_summary)
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)


for doc in cleaned_documents:
    print(f"{'-' * 40} {len(doc.page_content)} {'-' * 40}")
    print(doc.page_content)


---------------------------------------- 415 ----------------------------------------
A Comprehensive Guide to Post-Harvest Coffee Processing. Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices
---------------------------------------- 456 ----------------------------------------
The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-d

In [ ]:
import chromadb

collection_name = "recursive_chunking_documents"

# Set up ChromaDB client with persistent storage
chroma_client = chromadb.PersistentClient(path="../chroma_db")

# Delete existing collection if it exists (truncate)
try:
    chroma_client.delete_collection(name=collection_name)
    print("Existing collection deleted")
except:
    print("No existing collection found")

# Create new collection
print("Creating new collection...")
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "Document embeddings collection"}
)
print("Collection created successfully!")


17


In [ ]:
from langchain_openai import OpenAIEmbeddings

# Initialize embeddings model
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
# Calculate embeddings in batches and prepare data
documents = []
embeddings_list = []
ids = []

# Extract all texts first
all_texts = []
for idx, chunk in enumerate(chunks):
    text = chunk["page_content"] if isinstance(chunk, dict) else chunk.page_content
    all_texts.append(text)
    ids.append(f"doc_{idx}")

# Calculate embeddings in batches
batch_size = 100  # Adjust based on your needs and API limits
total_batches = (len(all_texts) + batch_size - 1) // batch_size

print(f"Processing {len(all_texts)} documents in {total_batches} batches...")

for i in range(0, len(all_texts), batch_size):
    batch_texts = all_texts[i:i + batch_size]
    
    # Calculate embeddings for the batch
    batch_embeddings = embeddings_model.embed_documents(batch_texts)
    
    # Add to main lists
    documents.extend(batch_texts)
    embeddings_list.extend(batch_embeddings)
    
    batch_num = (i // batch_size) + 1
    print(f"Processed batch {batch_num}/{total_batches} ({len(batch_texts)} documents)")

print(f"\nTotal embeddings calculated: {len(embeddings_list)}")

# Insert all data into ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids
)

print(f"Successfully added {len(documents)} documents to ChromaDB")

Processing 17 documents in 1 batches...
Processed batch 1/1 (17 documents)

Total embeddings calculated: 17
Successfully added 17 documents to ChromaDB


In [7]:
# Perform similarity search
query = """What are the environmental concerns and solutions related to coffee processing"""

# Calculate query embedding
query_embedding = embeddings_model.embed_query(query)

# Search for top K similar documents
k = 10  # Number of results to return

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=k
)

# Print results
print(f"Query: '{query}'\n")
print(f"Top {k} similar documents:\n")
print("=" * 80)

for i in range(len(results['ids'][0])):
    distance = results['distances'][0][i]
    
    # Correct conversion: cosine similarity = 1 - cosine distance
    # Then convert to percentage
    cosine_similarity = 1 - distance
    confidence_percentage = cosine_similarity * 100
    
    print(f"\nRank {i + 1}:")
    print(f"ID: {results['ids'][0][i]}")
    print(f"Raw Text: {results['documents'][0][i]}")
    print(f"Cosine Distance: {distance:.4f}")
    print(f"Cosine Similarity: {cosine_similarity:.4f}")
    print(f"Confidence: {confidence_percentage:.2f}%")
    print("-" * 80)

Query: 'What are the environmental concerns and solutions related to coffee processing'

Top 10 similar documents:


Rank 1:
ID: doc_3
Raw Text: .
Each method requires different amounts of water, time, and labor, which influences both the cost
of production and the environmental impact of coffee farming.
The global coffee industry processes approximately 10 million tons of coffee cherries annually, with
processing methods varying by region and producer size
Cosine Distance: 0.7804
Cosine Similarity: 0.2196
Confidence: 21.96%
--------------------------------------------------------------------------------

Rank 2:
ID: doc_1
Raw Text: . Different regions around the world have developed unique processing techniques based on
their climate, available resources, and traditional practices. The choice of processing method can
enhance certain flavor characteristics while suppressing others, making it one of the most critical
decisions in coffee production
Cosine Distance: 0.8134
Cosine Similari